# vLLM GPU benchmark

Select **Runtime → Change runtime type → GPU**, then run each cell in order. This experiment measures offline batch generation with a small public model. It does not benchmark Triton, a network gateway, or streaming TTFT. Record the GPU, model revision and software versions for every result. GPU availability and compatible packages vary.


In [ ]:
!nvidia-smi


Install vLLM in a fresh Colab runtime. If installation requests a restart, restart and run from the first cell. The install and model download require network access.


In [ ]:
%pip install -q vllm


In [ ]:
import torch, vllm, platform, json, time, statistics, subprocess, datetime
assert torch.cuda.is_available(), "Select a GPU runtime and reconnect."
print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda, "vLLM:", vllm.__version__)


In [ ]:
from vllm import LLM, SamplingParams
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# Record the resolved model revision separately if using a model that changes.
llm = LLM(model=MODEL, dtype="half", max_model_len=1024, gpu_memory_utilization=0.80, enforce_eager=True)
params = SamplingParams(temperature=0, max_tokens=64, ignore_eos=True)
prompts = [f"Explain a practical way to monitor model serving latency. Example {i}:" for i in range(8)]
llm.generate(prompts[:1], params)  # warmup


In [ ]:
results = []
for batch_size in (1, 4, 8):
    for trial in range(1, 4):
        start = time.perf_counter()
        outputs = llm.generate(prompts[:batch_size], params)
        seconds = time.perf_counter() - start
        generated = sum(len(out.outputs[0].token_ids) for out in outputs)
        row = {"batch_size": batch_size, "trial": trial, "seconds": seconds, "output_tokens": generated, "output_tokens_per_second": generated / seconds}
        results.append(row)
        print(row)


In [ ]:
report = {"experiment":"vLLM offline batch generation", "timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(), "model":MODEL, "model_revision":"not pinned; record resolved revision", "gpu":torch.cuda.get_device_name(0), "torch":torch.__version__, "cuda":torch.version.cuda, "vllm":vllm.__version__, "python":platform.python_version(), "settings":{"max_tokens":64,"temperature":0,"ignore_eos":True,"max_model_len":1024,"gpu_memory_utilization":0.80,"enforce_eager":True,"warmup_batches":1}, "limitations":"Offline batched calls; timing includes Python and tokenization overhead; no network, gateway, streaming or GPU memory measurements.", "results":results}
with open("vllm_gpu_results.json", "w") as f: json.dump(report, f, indent=2)
from google.colab import files
files.download("vllm_gpu_results.json")


Share the downloaded JSON and any errors. These results describe only this model, runtime and workload; do not generalize to production. The notebook has not yet been verified on your Colab GPU.
